### Step 1: Load Libraries

In [13]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display
from pprint import pprint
import json

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if OPENAI_API_KEY is None:
    raise Exception("API Key is missing!")
else:
    print("Key is: " + OPENAI_API_KEY[:8])

Key is: sk-proj-


### Step 2: Setup PushOver

In [14]:
load_dotenv()

pushover_user  = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url   = "https://api.pushover.net/1/messages.json"

print(pushover_user)
print(pushover_token)

Ut5xparzfvrod1rjygf36f7h1vqmx2
ahbf3s9qno83otx5dtgpr1bhknawj1


In [15]:
#Test Pushover
import requests

def send_notification(message: str):
    pay_load = {"user":pushover_user,
                "token": pushover_token,
                "message": message}
    requests.post(pushover_url, data = pay_load)

send_notification("Hello to Si Lam")
    

### Step 3: Describe Pushover in LLM

In [16]:
send_notification_function = {

    "name" : "send_notification",
    "description": "Send a push notification to user phone via pushover. Tell the user important task, event",
    "parameters": {
        "type": "object",
        "properties":{
            "message": {
                "type": "string",
                "description":"The notification message to send to user device"
            }
        },
        "required": ["message"]
    }
}

### Step 4: Add Pushover to the list of tools for LLM

In [17]:
tools = [
    {"type": "function","function": send_notification_function}
]

In [20]:
from email import message


def handle_tool_call(tool_calls):
    # return what to add to our context (about tool call results, a dictionary)
    tool_call = tool_calls[0] #assuming just 1 tool call
    args = json.loads(tool_call.function.arguments)

    send_notification(args['message'])
    print(f"Send notification: {args['message']}")

    tool_call_result  = {
        "role":"tool",
        "content":f"Notitication sent: {args['message']}",
        "tool_call_id":tool_call.id
        
    }
    return tool_call_result

### Step 5: calling the tool from LLM

In [26]:
client = OpenAI()
messages=[
        {"role":"user", "content":"Please send me a notification telling me what amzing progress I am making on the AI Engineering trainging by SuperDataScience"}
    ]
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages,
    tools= tools,
    tool_choice="auto"
)

#check if the model wants to call a tool

message = response.choices[0].message

if message.tool_calls:
    #handle tool call
    tool_result = handle_tool_call(message.tool_calls) #whole list of tools

    #add message to context
    messages.append(message)
    print("message appedn:" + str(message))

    #add info about tool call response to context, i.e. mesages
    messages.append(tool_result)
    print("tool_result:" + str(message))
    #invokde LLM one more time to get its updated response
    response = client.chat.completions.create(
        model='gpt-4.1-mini',
        messages=messages,
        #tools=tools #will be in the future
    )
    message = response.choices[0].message

    # print(message.content) from new LLM response
   
print(message.content)

Send notification: You are making amazing progress on the AI Engineering training by SuperDataScience! Keep up the great work!
message appedn:ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_yfSaBWpACiFjEKCoJPzmsJpH', function=Function(arguments='{"message":"You are making amazing progress on the AI Engineering training by SuperDataScience! Keep up the great work!"}', name='send_notification'), type='function')])
tool_result:ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_yfSaBWpACiFjEKCoJPzmsJpH', function=Function(arguments='{"message":"You are making amazing progress on the AI Engineering training by SuperDataScience! Keep up the great work!"}', name='send_notification'), type='function')])
You are making amazing progress on the AI En

In [ ]:
# if message.tool_calls:
#     tool_call = message.tool_calls[0]
#     args = json.loads(tool_call.function.arguments)

#     #Actually send notification

#     send_notification(args["message"])

#     print(f"Sent notification {args['message']}")

# else:
#     print(message.content)


Sent notification You are making amazing progress on the AI Engineering training by SuperDataScience! Keep up the great work!
